# Metadata export (S7)

Copies everything in the UniqueShip *12 Class - 5 Hours Each* bundle **except the audio**: the fold CSVs (`fold_*/metadata_5s_*.csv`) and any other small files. The dataset report (`scripts/dataset_report.py`) runs on these on the laptop.

**Input:** the `uniqueship-12-class-5-hours` dataset. CPU only, no internet.

**Output** in `/kaggle/working/uniqueship/`: the bundle's folder tree without the `.wav` files, plus `checksums.txt`.

## 1. Join the zip and extract everything but the audio

The bundle is a split zip, so the parts are joined first (`zip -s 0`). `unzip -x '*.wav'` then reads only the zip's table of contents and the small files, so it takes seconds, not the half hour of a full extraction.

In [ ]:
import glob
import hashlib
import os
import subprocess
import time
from pathlib import Path

hits = glob.glob("/kaggle/input/**/allclass_5hours.z01", recursive=True)
assert hits, "Attach the uniqueship-12-class-5-hours dataset (Add Input)."
D = os.path.dirname(hits[0])
assert os.path.exists(f"{D}/allclass_5hours.zip.part"), "Use a dataset version with both zip parts."
os.makedirs("/tmp/parts", exist_ok=True)
for src, dst in [("allclass_5hours.z01", "allclass_5hours.z01"),
                 ("allclass_5hours.zip.part", "allclass_5hours.zip")]:
    if not os.path.lexists(f"/tmp/parts/{dst}"):
        os.symlink(f"{D}/{src}", f"/tmp/parts/{dst}")

t = time.time()
subprocess.run("cd /tmp && rm -rf stage joined.zip && zip -q -s 0 parts/allclass_5hours.zip --out joined.zip"
               " && unzip -q joined.zip -x '*.wav' -d stage && rm joined.zip",
               shell=True, check=True)
print(f"extracted in {time.time() - t:.0f} s")

## 2. Copy to the output and check

Copies the bundle folder (the parent of `fold_1`) to `/kaggle/working/uniqueship/`, lists every file with its size, checks that all 5 folds have their train, validation and test CSVs, and writes a SHA-256 of every file to `checksums.txt`.

In [ ]:
import shutil

fold_dirs = glob.glob("/tmp/stage/**/fold_1", recursive=True)
assert len(fold_dirs) == 1, fold_dirs
BUNDLE = Path(fold_dirs[0]).parent
OUT = Path("/kaggle/working/uniqueship")
shutil.rmtree(OUT, ignore_errors=True)
shutil.copytree(BUNDLE, OUT)

files = sorted(p for p in OUT.rglob("*") if p.is_file())
for p in files:
    print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(OUT)}")

missing = [f"fold_{k}/metadata_5s_{s}.csv" for k in range(1, 6)
           for s in ("train", "validation", "test") if not (OUT / f"fold_{k}/metadata_5s_{s}.csv").exists()]
assert not missing, f"Missing: {missing}"

lines = [f"{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.relative_to(OUT)}" for p in files]
(OUT / "checksums.txt").write_text("\n".join(lines) + "\n")
print(f"\n{len(files)} files, all 15 fold CSVs present")

## After the run

On the laptop: `kaggle kernels output yanyetang/acousticwise-metadata-export -p data/` then, inside `data/uniqueship/`, `shasum -a 256 -c checksums.txt`.